# BTW 2025: Building Constituency Vote Entries

This notebook mirrors the step-by-step 2021 preparation workflow. It turns the official 2025 polling-district results and representative election statistics into the detailed `VoteEntry` rows used by the application.

The important distinction is visible throughout the notebook:

- the polling-district file supplies the exact constituency, party, vote-type and postal/in-person totals;
- the representative statistics supply demographic proportions at state level;
- the final constituency-level demographic cells are therefore modelled distributions constrained by those official totals.

The notebook deliberately shows the source tables, normalization decisions, reshaped data, fitted profiles and conservation checks instead of hiding them behind one pipeline call.

For 2025 the published gender category `m|d|o` includes male, diverse and people without a gender entry. The shared JSON value remains `gender="m"`, but it retains that broader source meaning.

## 0. Working directory and local files

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("The notebook must run inside the repository folder.")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
# These names are local suggestions. Point them at the files you downloaded.
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw25_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw25_rws_bst2.csv"
OUTPUT_DIRECTORY = ROOT / "scripts/data/generated/btw2025"

DISTRICT_RESULTS_CSV, STATE_DEMOGRAPHICS_CSV, OUTPUT_DIRECTORY

In [ ]:
from scripts.election_data.btw2025 import (
    BTW2025_AGE_GROUPS,
    district_party_columns,
    normalize_state_statistic_rows,
    read_polling_district_csv,
    read_representative_statistics_csv,
    reshape_polling_district_votes,
)
from scripts.election_data.btw2025_statistics import reshape_state_statistic_votes
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    calculate_demographic_profiles,
    inspect_district_rows,
    normalize_district_rows,
    select_state_statistic_detail_rows,
    select_usable_district_rows,
)
from scripts.election_data.pipeline import distribute_district_votes, write_vote_entries
from scripts.election_data.profiles import build_state_method_profiles
from scripts.election_data.validation import entries_to_frame, validate_vote_entries

## 1. Read the polling-district file as published

The 2025 CSV starts with a metadata preamble. The reader searches for the actual row beginning with `Wahlkreis;Land` and only then lets pandas parse the table. No election row is removed or normalized in this step.

In [ ]:
raw_districts = read_polling_district_csv(DISTRICT_RESULTS_CSV)

print(f"Rows: {len(raw_districts):,}")
print(f"Columns: {len(raw_districts.columns):,}")
print("First columns:", raw_districts.columns[:18].tolist())
print("Last columns:", raw_districts.columns[-12:].tolist())

display(raw_districts.head())
display(raw_districts.tail())

## 2. Inspect constituency identifiers before removing anything

As in the 2021 notebook, empty footer rows may be removable, but non-empty unexpected constituency values must not disappear silently. The diagnostic table classifies every row first.

In [ ]:
district_diagnostics = inspect_district_rows(raw_districts)

display(district_diagnostics["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))
display(district_diagnostics[district_diagnostics["status"] != "usable"].head(30))

invalid_rows = district_diagnostics[district_diagnostics["status"] == "invalid"]
if not invalid_rows.empty:
    raise ValueError("Unexpected non-empty constituency identifiers are present; inspect them above before continuing.")

## 3. Keep usable rows and normalize constituency, state and election method

`Bezirksart == 5` becomes `postal`. The documented polling-place categories `0`, `6` and `8` become `in-person`, matching the two-value application model. State codes are converted to the same state names used by the frontend.

In [ ]:
usable_districts = select_usable_district_rows(raw_districts, district_diagnostics)
normalized_districts = normalize_district_rows(usable_districts)

print(f"Usable polling-district rows: {len(normalized_districts):,}")
print(f"Constituencies: {normalized_districts['districtId'].nunique()}")
print(f"States: {normalized_districts['state'].nunique()}")

display(normalized_districts[["Wahlkreis", "Land", "Bezirksart", "districtId", "state", "electionMethod"]].head(20))
display(normalized_districts["electionMethod"].value_counts().rename_axis("electionMethod").to_frame("polling-district rows"))

## 4. Discover the first- and second-vote party columns

Unlike 2021, the 2025 source uses labels such as `SPD - Erststimmen` and `SPD - Zweitstimmen`. The helper selects those suffixes and explicitly excludes the aggregate `Gültige` and `Ungültige` columns. The discovered columns are shown before reshaping.

In [ ]:
first_party_columns = district_party_columns(normalized_districts, "1")
second_party_columns = district_party_columns(normalized_districts, "2")

print(f"First-vote party columns: {len(first_party_columns)}")
print(f"Second-vote party columns: {len(second_party_columns)}")
print("First vote:", first_party_columns)
print("Second vote:", second_party_columns)

## 5. Reshape polling-district party columns into vote rows

The wide source table is converted into one row per polling district, party and vote type. At this point the rows still retain the polling-district source row and election method; no demographic model has been applied.

In [ ]:
first_polling_votes = reshape_polling_district_votes(normalized_districts, vote_type="1")
second_polling_votes = reshape_polling_district_votes(normalized_districts, vote_type="2")

print(f"Long first-vote rows: {len(first_polling_votes):,}")
print(f"Long second-vote rows: {len(second_polling_votes):,}")
display(first_polling_votes.head(20))
display(second_polling_votes.head(20))

display(pd.DataFrame({
    "voteType": ["1", "2"],
    "valid party votes represented": [first_polling_votes["votes"].sum(), second_polling_votes["votes"].sum()],
}))

## 6. Aggregate polling districts to constituencies

The application does not store polling districts. The exact source totals are therefore aggregated to `constituency × state × party × vote type × election method`. These totals become the fixed margins that every later modelling step must preserve.

In [ ]:
first_district_totals = aggregate_to_constituencies(first_polling_votes)
second_district_totals = aggregate_to_constituencies(second_polling_votes)
district_totals = pd.concat([first_district_totals, second_district_totals], ignore_index=True)

print(f"First-vote constituency/method groups: {len(first_district_totals):,}")
print(f"Second-vote constituency/method groups: {len(second_district_totals):,}")

sample_district = int(district_totals["districtId"].min())
display(
    district_totals[district_totals["districtId"] == sample_district]
    .sort_values(["voteType", "party", "electionMethod"])
    .head(40)
)

## 7. Read the representative election statistics as published

The statistics file uses `#` comment lines before the semicolon-separated table. This step only reads the table and displays the source categories before any normalization. In particular, the original `Geschlecht` and `Geburtsjahresgruppe` values remain visible here.

In [ ]:
raw_statistics = read_representative_statistics_csv(STATE_DEMOGRAPHICS_CSV)

print(f"Rows: {len(raw_statistics):,}")
print(f"Columns: {len(raw_statistics.columns):,}")
print("Published gender values:", raw_statistics["Geschlecht"].drop_duplicates().tolist())
print("Published birth-year groups:", raw_statistics["Geburtsjahresgruppe"].drop_duplicates().tolist())
print("Published vote-type values:", raw_statistics["Erst-/Zweitstimme"].drop_duplicates().tolist())

display(raw_statistics.head(20))
display(raw_statistics.tail(20))

## 8. Normalize the statistical dimensions, but keep summary rows visible

The source dimensions are mapped to the shared fields used downstream. Summary rows deliberately remain in the frame so the notebook can show what will later be excluded.

For 2025, `m|d|o` maps to the internal value `m`. The published birth-year cohorts map to the 2025 age-group labels used by this preparation workflow; they are not split into the different 2021 boundaries.

In [ ]:
normalized_statistics = normalize_state_statistic_rows(raw_statistics)

gender_mapping = normalized_statistics[["Geschlecht", "gender"]].drop_duplicates().sort_values("Geschlecht")
age_mapping = normalized_statistics[["Geburtsjahresgruppe", "ageGroup"]].drop_duplicates()

display(gender_mapping)
display(age_mapping)

normalized_dimensions = ["state", "voteType", "gender", "ageGroup"]
summary_rows = normalized_statistics[normalized_statistics[normalized_dimensions].isna().any(axis=1)]
print(f"Rows containing at least one summary dimension: {len(summary_rows):,}")
display(summary_rows.head(30))

## 9. Select state × vote type × gender × age detail rows

Only rows with all four normalized dimensions are demographic detail observations. This selection removes national totals and `Summe` rows only after they have been inspected above.

In [ ]:
detail_statistics = select_state_statistic_detail_rows(normalized_statistics)

print(f"Demographic detail rows: {len(detail_statistics):,}")
print(f"States represented: {detail_statistics['state'].nunique()}")
print("Age groups:", detail_statistics["ageGroup"].drop_duplicates().tolist())
print("Genders:", detail_statistics["gender"].drop_duplicates().tolist())
display(detail_statistics.head(30))

## 10. Reshape the published party statistics

The party columns are converted to long rows. The 2025-specific reshaper also normalizes party identifiers such as `Die Linke` to the application's established `DIE LINKE` value. `dar.` columns are retained as their named party profiles, while `Sonstige` remains available as a fallback profile for parties without their own demographic column.

In [ ]:
statistic_votes = reshape_state_statistic_votes(detail_statistics)

print(f"Long statistic rows: {len(statistic_votes):,}")
print("Parties/categories:", sorted(statistic_votes["party"].unique()))
display(statistic_votes.head(30))

sample_state = sorted(statistic_votes["state"].unique())[0]
display(
    statistic_votes[(statistic_votes["state"] == sample_state) & (statistic_votes["voteType"] == "2")]
    .sort_values(["party", "gender", "ageGroup"])
    .head(40)
)

## 11. Convert rounded statistical counts into demographic profiles

The representative statistics are not used as competing official vote totals. For each state, vote type and party/category, their published counts are converted into shares across the demographic cells. Each available profile should therefore sum to 1.

In [ ]:
demographic_profiles = calculate_demographic_profiles(statistic_votes)

profile_sums = demographic_profiles.groupby(["state", "voteType", "party"], as_index=False)["share"].sum()
profile_sums["distanceFromOne"] = (profile_sums["share"] - 1.0).abs()

display(profile_sums.sort_values("distanceFromOne", ascending=False).head(20))
print("Maximum profile-sum error:", profile_sums["distanceFromOne"].max())

sample_profile = demographic_profiles[
    (demographic_profiles["state"] == sample_state)
    & (demographic_profiles["voteType"] == "2")
    & (demographic_profiles["party"] == "SPD")
]
display(sample_profile[["gender", "ageGroup", "statisticVotes", "share"]])

## 12. Fit demographic profiles to the exact postal/in-person totals

For every state, vote type and party, iterative proportional fitting combines two known margins:

- demographic shares from the representative statistics;
- exact `in-person` and `postal` totals from the polling-district results.

There is no separate 2025 federal method-by-demographic seed file in this workflow, so the initial table assumes independence between demographics and election method. IPF then adjusts it until both margins agree. The fitted state profile is later applied to every constituency of that state.

In [ ]:
profiles = build_state_method_profiles(
    district_totals,
    demographic_profiles,
    age_groups=BTW2025_AGE_GROUPS,
)

print(f"Fitted profile rows: {len(profiles):,}")
display(profiles["demographicProfileSource"].value_counts().rename_axis("source").to_frame("rows"))
display(profiles["methodSeedSource"].value_counts().rename_axis("source").to_frame("rows"))

sample_fit = profiles[
    (profiles["state"] == sample_state)
    & (profiles["voteType"] == "2")
    & (profiles["party"] == "SPD")
]
display(sample_fit[["electionMethod", "gender", "ageGroup", "share", "fittedStateVotes", "demographicProfileSource", "methodSeedSource"]])

## 13. Apply the fitted state profiles to each constituency

Each exact constituency/party/vote-type/election-method total is multiplied by the corresponding fitted state demographic shares. Fractional values are expected: these rows are statistical weights, not reconstructed individual ballots.

The example below displays both the detailed cells and their sum so the multiplication can be inspected directly.

In [ ]:
entries = distribute_district_votes(district_totals, profiles)
entry_frame = entries_to_frame(entries)

print(f"Generated detail rows: {len(entry_frame):,}")

example_source = district_totals.sort_values(["districtId", "voteType", "party", "electionMethod"]).iloc[0]
example_mask = (
    (entry_frame["districtId"] == example_source["districtId"])
    & (entry_frame["state"] == example_source["state"])
    & (entry_frame["party"] == example_source["party"])
    & (entry_frame["voteType"] == example_source["voteType"])
    & (entry_frame["electionMethod"] == example_source["electionMethod"])
)
example_detail = entry_frame[example_mask].sort_values(["gender", "ageGroup"])

display(example_source.to_frame("official source group"))
display(example_detail)
print("Official total:", example_source["votes"])
print("Generated demographic sum:", example_detail["votes"].sum())

## 14. Validate conservation before writing files

The validation checks two things independently for each vote type:

1. demographic cells sum back to every exact constituency/party/election-method source total;
2. the generated constituency rows sum back to the fitted state demographic margins.

A failed check stops the notebook before output is written.

In [ ]:
first_entries = [entry for entry in entries if entry.voteType == "1"]
second_entries = [entry for entry in entries if entry.voteType == "2"]

first_report = validate_vote_entries(
    first_entries,
    first_district_totals,
    profiles[profiles["voteType"] == "1"],
)
second_report = validate_vote_entries(
    second_entries,
    second_district_totals,
    profiles[profiles["voteType"] == "2"],
)

display(first_report)
display(second_report)

## 15. Inspect nationwide and demographic summaries

Before writing the JSON, aggregate the generated rows back up. These tables do not replace the exact conservation checks above; they are a human-readable plausibility view similar to the 2021 notebook.

In [ ]:
nationwide = entry_frame.groupby(["voteType", "party"], as_index=False)["votes"].sum()
nationwide["percentage"] = nationwide["votes"] / nationwide.groupby("voteType")["votes"].transform("sum") * 100

print("Largest parties by first vote")
display(nationwide[nationwide["voteType"] == "1"].sort_values("votes", ascending=False).head(20))
print("Largest parties by second vote")
display(nationwide[nationwide["voteType"] == "2"].sort_values("votes", ascending=False).head(20))

demographic_totals = entry_frame.groupby(["voteType", "gender", "ageGroup"], as_index=False)["votes"].sum()
display(demographic_totals.sort_values(["voteType", "gender", "ageGroup"]))

## 16. Write the generated JSON files

Only after the source inspection, transformations and validation above do we write the two application datasets. The second notebook re-opens these finished files and validates them independently.

In [ ]:
first_path = write_vote_entries(first_entries, OUTPUT_DIRECTORY / "first_votes.json")
second_path = write_vote_entries(second_entries, OUTPUT_DIRECTORY / "second_votes.json")

print(first_path)
print(second_path)

Run `02_validate_btw2025_vote_entries.ipynb` next. It treats the generated JSON as finished artifacts and checks file size, structure, category coverage, duplicates, example constituencies, nationwide percentages and reconstruction of the official source totals.